# Taller 2 · Listas de espera: ¿cómo varían por provincia y especialidad?

INSTRUCCIONES PARA EL ALUMNADO:  
1. Descarga el CSV del dataset de Lista de espera quirurgica (Catalogo de  
   Informacion Publica, Consejeria de Sanidad, JCyL).  
2. Guardalo como 'lista_espera_quirurgica.csv' en la carpeta 'datos/' de este taller.  
3. Ejecuta las celdas en orden. Los CHECKPOINT verifican que vas bien.  


In [ ]:
# Celda 1 - Importar librerias
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Celda 2 - Cargar el dataset
try:
    df = pd.read_csv('datos/lista_espera_quirurgica.csv', sep=';', encoding='utf-8')
except FileNotFoundError:
    raise FileNotFoundError(
        "No se encuentra 'datos/lista_espera_quirurgica.csv'. "
        "Descargalo primero y colocalo en la carpeta 'datos/' de este taller."
    )

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

In [ ]:
# Celda 3 - Exploracion inicial (Actividad 1)
print("Columnas:", df.columns.tolist())
print("\nEspecialidades:", sorted(df['especialidad'].unique()))
print("\nProvincias:", sorted(df['provincia'].unique()))
print(f"\nNumero de especialidades distintas: {df['especialidad'].nunique()}")

In [ ]:
# CHECKPOINT 1
assert df['provincia'].nunique() <= 9, "Revisa nombres de provincia, puede haber duplicados por errores de escritura"
assert df['tiempo_medio_espera_dias'].min() >= 0, "Hay tiempos de espera negativos, revisa el dataset"
print("Checkpoint OK: datos con formato coherente")

In [ ]:
# Celda 4 - Tiempo medio de espera por especialidad en CyL (Actividad 2)
por_especialidad = (
    df.groupby('especialidad')['tiempo_medio_espera_dias']
    .mean()
    .sort_values(ascending=False)
)

print("Tiempo medio de espera por especialidad (dias):")
print(por_especialidad)

plt.figure(figsize=(9, 6))
por_especialidad.plot(kind='barh', color='indianred')
plt.xlabel('Tiempo medio de espera (dias)')
plt.title('Tiempo medio de espera quirurgica por especialidad - Castilla y Leon')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('espera_por_especialidad.png', dpi=150)
plt.show()

In [ ]:
# Celda 5 - Identificar top-3 especialidades con mayor espera
top3 = por_especialidad.head(3).index.tolist()
print(f"Las 3 especialidades con mayor tiempo de espera son: {top3}")

In [ ]:
# Celda 6 - Comparativa por provincia para las top-3 especialidades (Actividad 3)
comparativa = (
    df[df['especialidad'].isin(top3)]
    .pivot_table(index='provincia', columns='especialidad', values='tiempo_medio_espera_dias')
)

comparativa.plot(kind='bar', figsize=(11, 6))
plt.ylabel('Tiempo medio de espera (dias)')
plt.title('Comparativa de tiempo de espera por provincia y especialidad')
plt.legend(title='Especialidad', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('comparativa_provincia_especialidad.png', dpi=150)
plt.show()

In [ ]:
# Celda 7 - Provincia con mayor y menor espera por especialidad
for especialidad in top3:
    datos_esp = comparativa[especialidad].dropna()
    print(f"\n{especialidad}:")
    print(f"  Mayor espera: {datos_esp.idxmax()} ({datos_esp.max():.1f} dias)")
    print(f"  Menor espera: {datos_esp.idxmin()} ({datos_esp.min():.1f} dias)")

In [ ]:
# Celda 8 - Comparacion con media nacional (Actividad 4)
media_nacional_dias = 115  # <-- ACTUALIZAR con dato oficial vigente del SISNS

media_cyl_dias = df['tiempo_medio_espera_dias'].mean()
diferencia = media_cyl_dias - media_nacional_dias

print(f"Media de Castilla y Leon: {media_cyl_dias:.1f} dias")
print(f"Media nacional de referencia (SISNS): {media_nacional_dias} dias")
print(f"Diferencia: {diferencia:+.1f} dias ({'por encima' if diferencia > 0 else 'por debajo'} de la media nacional)")

In [ ]:
# EJERCICIO - Completa tu mismo
# Elige una especialidad que NO este en el top-3 y repite el analisis de la Celda 6.
# ¿Su comportamiento por provincia es similar o distinto al de las especialidades con mas espera?